# Chapter 12: Measurement Infrastructure and Attribution - Code Validation

This notebook tests the code examples from Chapter 12 to ensure they are production-quality and executable.

## Table of Contents
1. Incremental Event Graph Construction (Code Listing 12.1)
2. User Journey Replay — The Example from Section 1.2
3. Eviction Rules — VIEW_LOOKBACK and CLICK_LOOKBACK
4. View-Through Attribution Window
5. Last-Touch Attribution Priority (Click > Impression)
6. Shapley Value Attribution (Code Listings 12.2 & 12.3)
7. PyFlink Streaming Attribution Join (Section 2.2)
8. Inverse Propensity Scoring (Code Listing 12.4)
9. Pipeline Health Monitoring (Code Listing 12.5)

## 1. Incremental Event Graph Construction

Testing Code Listing 12.1: the `AttributionGraphStore` prototype that maintains per-user touchpoint graphs in memory with incremental updates on each impression, click, and conversion event.

In [2]:
from dataclasses import dataclass, field
from collections import defaultdict
from datetime import datetime, timedelta

CLICK_LOOKBACK = timedelta(days=14)          # click-through attribution window
VIEW_LOOKBACK = timedelta(days=1)            # view-through attribution window


@dataclass
class Touchpoint:
    impression_id: str
    campaign_id: str
    creative_id: str
    impression_time: datetime
    click_time: datetime | None = None


@dataclass
class AttributionRecord:
    conversion_id: str
    touchpoint: Touchpoint
    credit: float


@dataclass
class UserGraph:
    """Per-user touchpoint graph, keyed by impression_id for O(1) click joins."""
    nodes: dict[str, Touchpoint] = field(default_factory=dict)

    def evict(self, now: datetime) -> None:
        """Remove stale nodes: unclicked impressions past VIEW_LOOKBACK,
        and clicked touchpoints past CLICK_LOOKBACK."""
        expired = [
            imp_id for imp_id, tp in self.nodes.items()
            if (tp.click_time is None and now - tp.impression_time > VIEW_LOOKBACK)
            or (tp.click_time is not None and now - tp.click_time > CLICK_LOOKBACK)
        ]
        for imp_id in expired:
            del self.nodes[imp_id]


class AttributionGraphStore:
    """In-memory store of per-user attribution graphs."""

    def __init__(self) -> None:
        self.graphs: dict[str, UserGraph] = defaultdict(UserGraph)

    def on_impression(self, user_id: str, impression_id: str,
                      campaign_id: str, creative_id: str,
                      timestamp: datetime) -> None:
        """An impression creates a new touchpoint node."""
        graph = self.graphs[user_id]
        graph.nodes[impression_id] = Touchpoint(
            impression_id=impression_id,
            campaign_id=campaign_id,
            creative_id=creative_id,
            impression_time=timestamp,
        )
        graph.evict(now=timestamp)

    def on_click(self, user_id: str, impression_id: str,
                 timestamp: datetime) -> None:
        """A click attaches to its parent impression via impression_id."""
        graph = self.graphs[user_id]
        tp = graph.nodes.get(impression_id)
        if tp is not None:
            tp.click_time = timestamp

    def on_conversion(self, user_id: str, conversion_id: str,
                      timestamp: datetime) -> list[AttributionRecord]:
        """A conversion scans the user's graph to resolve attribution."""
        graph = self.graphs[user_id]
        graph.evict(now=timestamp)

        qualifying: list[Touchpoint] = [
            tp for tp in graph.nodes.values()
            if (tp.click_time is not None
                and timestamp - tp.click_time <= CLICK_LOOKBACK)
            or (tp.click_time is None
                and timestamp - tp.impression_time <= VIEW_LOOKBACK)
        ]
        return self._last_touch(conversion_id, qualifying)

    @staticmethod
    def _last_touch(conversion_id: str,
                    touchpoints: list[Touchpoint]) -> list[AttributionRecord]:
        """Last-touch: 100% credit to most recent click, or impression."""
        if not touchpoints:
            return []
        best = max(
            touchpoints,
            key=lambda tp: (tp.click_time or datetime.min, tp.impression_time),
        )
        return [AttributionRecord(conversion_id, best, credit=1.0)]

print("AttributionGraphStore loaded successfully.")

AttributionGraphStore loaded successfully.


## 2. User Journey Replay

Replaying the example journey from Section 1.2:

```
Display Impression (t-72h)
  → Search Impression (t-24h)
  → Search Click (t-23h 59m)
  → Display Impression (t-2h)
  → Display Click (t-1h 59m)
  → Purchase (t=0)
```

We expect last-touch attribution to credit the **Display Click** (the most recent click before purchase).

In [3]:
# Replay the Section 1.2 user journey
store = AttributionGraphStore()

t_purchase = datetime(2026, 3, 8, 12, 0, 0)
user = "user_42"

# Display Impression (t-72h) — upper-funnel, no click
store.on_impression(user, "imp_display_1", "camp_display", "creative_banner_a",
                    t_purchase - timedelta(hours=72))

# Search Impression (t-24h)
store.on_impression(user, "imp_search_1", "camp_search", "creative_search_a",
                    t_purchase - timedelta(hours=24))

# Search Click (t-23h 59m) — clicks the search impression ~1 min later
store.on_click(user, "imp_search_1",
               t_purchase - timedelta(hours=23, minutes=59))

# Display Impression (t-2h)
store.on_impression(user, "imp_display_2", "camp_display", "creative_banner_b",
                    t_purchase - timedelta(hours=2))

# Display Click (t-1h 59m) — clicks the display impression ~1 min later
store.on_click(user, "imp_display_2",
               t_purchase - timedelta(hours=1, minutes=59))

# Purchase (t=0)
records = store.on_conversion(user, "conv_001", t_purchase)

print(f"Attribution records: {len(records)}")
for r in records:
    tp = r.touchpoint
    print(f"  credit={r.credit:.1f}  impression={tp.impression_id}  "
          f"campaign={tp.campaign_id}  click_time={tp.click_time}")

# Verify: last-touch should credit the display click (most recent click)
assert len(records) == 1
assert records[0].touchpoint.impression_id == "imp_display_2"
assert records[0].credit == 1.0
print("\n✓ Last-touch correctly attributed to most recent click (Display Click at t-1h 59m)")

Attribution records: 1
  credit=1.0  impression=imp_display_2  campaign=camp_display  click_time=2026-03-08 10:01:00

✓ Last-touch correctly attributed to most recent click (Display Click at t-1h 59m)


## 3. Eviction Rules

Testing the two eviction mechanisms:
1. **VIEW_LOOKBACK** (1 day): Unclicked impressions are dropped once they age past the view-through window.
2. **CLICK_LOOKBACK** (14 days): Clicked impression/click pairs are dropped once they age past the click-through window.

In [5]:
# --- Test 3a: VIEW_LOOKBACK eviction ---
# An unclicked impression older than 1 day should be evicted.

store = AttributionGraphStore()
t0 = datetime(2026, 3, 8, 12, 0, 0)
user = "user_evict"

# Impression at t=0, no click
store.on_impression(user, "imp_stale", "camp_a", "creative_a", t0)

# Verify it exists
assert "imp_stale" in store.graphs[user].nodes
print(f"After impression: {len(store.graphs[user].nodes)} node(s) — imp_stale present")

# 25 hours later, a new impression arrives (triggers eviction)
store.on_impression(user, "imp_fresh", "camp_b", "creative_b",
                    t0 + timedelta(hours=25))

assert "imp_stale" not in store.graphs[user].nodes, "imp_stale should have been evicted"
assert "imp_fresh" in store.graphs[user].nodes
print(f"After 25h: {len(store.graphs[user].nodes)} node(s) — imp_stale evicted, imp_fresh present")

# Verify that an unclicked impression within 1 day is NOT evicted
store.on_impression(user, "imp_recent", "camp_c", "creative_c",
                    t0 + timedelta(hours=25, minutes=30))
assert "imp_fresh" in store.graphs[user].nodes, "imp_fresh should still be present (only 30 min old)"
print(f"After 25h 30m: {len(store.graphs[user].nodes)} node(s) — imp_fresh still present (only 30m old)")
print("\n✓ VIEW_LOOKBACK eviction works: unclicked impressions dropped after 1 day, kept within window")

After impression: 1 node(s) — imp_stale present
After 25h: 1 node(s) — imp_stale evicted, imp_fresh present
After 25h 30m: 2 node(s) — imp_fresh still present (only 30m old)

✓ VIEW_LOOKBACK eviction works: unclicked impressions dropped after 1 day, kept within window


In [6]:
# --- Test 3b: CLICK_LOOKBACK eviction ---
# A clicked touchpoint older than 14 days should be evicted.

store = AttributionGraphStore()
t0 = datetime(2026, 1, 1, 12, 0, 0)
user = "user_old_click"

# Impression + click at t=0
store.on_impression(user, "imp_old", "camp_a", "creative_a", t0)
store.on_click(user, "imp_old", t0 + timedelta(seconds=30))

assert "imp_old" in store.graphs[user].nodes
print(f"After click: {len(store.graphs[user].nodes)} node(s) — imp_old with click present")

# 15 days later, a conversion arrives (triggers eviction before attribution)
t_conv = t0 + timedelta(days=15)
records = store.on_conversion(user, "conv_late", t_conv)

assert len(records) == 0, "No attribution: the clicked touchpoint is past CLICK_LOOKBACK"
assert "imp_old" not in store.graphs[user].nodes, "imp_old should have been evicted"
print(f"After 15 days: {len(store.graphs[user].nodes)} node(s) — imp_old evicted")
print(f"Attribution records: {len(records)} (no qualifying touchpoints)")
print("\n✓ CLICK_LOOKBACK eviction works: clicked pairs dropped after 14 days")

After click: 1 node(s) — imp_old with click present
After 15 days: 0 node(s) — imp_old evicted
Attribution records: 0 (no qualifying touchpoints)

✓ CLICK_LOOKBACK eviction works: clicked pairs dropped after 14 days


## 4. View-Through Attribution Window

Testing that an unclicked impression within `VIEW_LOOKBACK` (1 day) qualifies for
view-through attribution, and one outside the window does not.

In [7]:
# --- Test 4a: View-through within 1 day — should attribute ---
store = AttributionGraphStore()
t_conv = datetime(2026, 3, 8, 12, 0, 0)
user = "user_view"

# Impression 20 hours ago, no click (within VIEW_LOOKBACK = 1 day)
# But we need it within IMPRESSION_TTL too, so we set a fresh impression
# Note: IMPRESSION_TTL is 30 min, but VIEW_LOOKBACK is 1 day.
# In production, eviction only runs on new events. Let's simulate a scenario
# where the impression is 20 hours old but wasn't evicted because no new
# events arrived in between. We insert it directly to test the attribution logic.

store.graphs[user].nodes["imp_view"] = Touchpoint(
    impression_id="imp_view",
    campaign_id="camp_display",
    creative_id="creative_banner",
    impression_time=t_conv - timedelta(hours=20),
    click_time=None,
)

records = store.on_conversion(user, "conv_view", t_conv)

assert len(records) == 1
assert records[0].touchpoint.impression_id == "imp_view"
assert records[0].touchpoint.click_time is None
print(f"View-through attribution: credit={records[0].credit:.1f} "
      f"impression={records[0].touchpoint.impression_id}")
print("\n✓ Unclicked impression within VIEW_LOOKBACK (20h < 1 day) qualifies for attribution")

View-through attribution: credit=1.0 impression=imp_view

✓ Unclicked impression within VIEW_LOOKBACK (20h < 1 day) qualifies for attribution


In [8]:
# --- Test 4b: View-through outside 1 day — should NOT attribute ---
store = AttributionGraphStore()
t_conv = datetime(2026, 3, 8, 12, 0, 0)
user = "user_view_expired"

# Impression 25 hours ago, no click (outside VIEW_LOOKBACK = 1 day)
store.graphs[user].nodes["imp_old_view"] = Touchpoint(
    impression_id="imp_old_view",
    campaign_id="camp_display",
    creative_id="creative_banner",
    impression_time=t_conv - timedelta(hours=25),
    click_time=None,
)

records = store.on_conversion(user, "conv_no_view", t_conv)

assert len(records) == 0
print(f"Attribution records: {len(records)} (impression too old for view-through)")
print("\n✓ Unclicked impression outside VIEW_LOOKBACK (25h > 1 day) does NOT qualify")

Attribution records: 0 (impression too old for view-through)

✓ Unclicked impression outside VIEW_LOOKBACK (25h > 1 day) does NOT qualify


## 5. Last-Touch Priority: Click > Impression

Verifying that last-touch attribution prefers a click over a more recent impression.
This handles the case where a user clicks an ad, then sees a different ad impression
just before converting — credit should go to the click, not the later impression.

In [9]:
store = AttributionGraphStore()
t_conv = datetime(2026, 3, 8, 12, 0, 0)
user = "user_priority"

# Click happened 2 hours ago
store.graphs[user].nodes["imp_clicked"] = Touchpoint(
    impression_id="imp_clicked",
    campaign_id="camp_search",
    creative_id="creative_search",
    impression_time=t_conv - timedelta(hours=2, minutes=1),
    click_time=t_conv - timedelta(hours=2),
)

# A different impression (no click) happened 30 seconds before purchase
store.graphs[user].nodes["imp_recent_view"] = Touchpoint(
    impression_id="imp_recent_view",
    campaign_id="camp_display",
    creative_id="creative_banner",
    impression_time=t_conv - timedelta(seconds=30),
    click_time=None,
)

records = store.on_conversion(user, "conv_priority", t_conv)

assert len(records) == 1
assert records[0].touchpoint.impression_id == "imp_clicked"
print(f"Attributed to: {records[0].touchpoint.impression_id} "
      f"(click at {records[0].touchpoint.click_time})")
print(f"NOT attributed to: imp_recent_view "
      f"(impression only, 30s before conversion)")
print("\n✓ Last-touch correctly prioritizes click over more recent impression")

Attributed to: imp_clicked (click at 2026-03-08 10:00:00)
NOT attributed to: imp_recent_view (impression only, 30s before conversion)

✓ Last-touch correctly prioritizes click over more recent impression


## 6. Value Function Training & Shapley Attribution (Code Listings 12.2 & 12.3)

Testing the end-to-end Shapley pipeline:
1. **Code Listing 12.2** — Train a `value_fn` from historical journey data using logistic regression
2. **Code Listing 12.3** — Compute Shapley values using the trained `value_fn`

We also verify the Shapley axioms (efficiency, null player, symmetry) with hand-crafted value functions.

In [15]:
# --- Code Listing 12.2: Train the value function from historical journeys ---
from sklearn.linear_model import LogisticRegression

CHANNELS = ["search", "display", "browse", "email"]

def encode_journey(channels_present: frozenset[str]) -> np.ndarray:
    """Encode a set of channels as a binary feature vector."""
    return np.array([1.0 if ch in channels_present else 0.0
                     for ch in CHANNELS])

def train_value_function(
    journeys: list[tuple[frozenset[str], int]],
) -> "Callable[[frozenset[str]], float]":
    """Train a logistic regression model to estimate P(conversion | channels).

    Args:
        journeys: List of (channels_exposed, converted) tuples.

    Returns:
        A value function v(S) -> float mapping a channel coalition
        to predicted conversion probability.
    """
    X = np.array([encode_journey(ch) for ch, _ in journeys])
    y = np.array([label for _, label in journeys])
    model = LogisticRegression(max_iter=1000).fit(X, y)

    def value_fn(coalition: frozenset[str]) -> float:
        if len(coalition) == 0:
            return 0.0
        x = encode_journey(coalition).reshape(1, -1)
        return float(model.predict_proba(x)[0, 1])

    return value_fn

print("train_value_function loaded successfully.")

<frozen importlib._bootstrap>:488: RuntimeWarning: The global interpreter lock (GIL) has been enabled to load module 'sklearn.__check_build._check_build', which has not declared that it can run safely without the GIL. To override this behavior and keep the GIL disabled (at your own risk), run with PYTHON_GIL=0 or -Xgil=0.


train_value_function loaded successfully.


In [16]:
# --- Test 6e: End-to-end pipeline — train value_fn, then compute Shapley ---
# Generate synthetic historical journey data where:
#   - search has a strong lift on conversion
#   - display has moderate lift
#   - browse has weak lift
#   - email has minimal lift
np.random.seed(42)
N = 5000
synthetic_journeys: list[tuple[frozenset[str], int]] = []

for _ in range(N):
    # Random channel exposure
    exposed = {ch for ch in CHANNELS if np.random.rand() < 0.4}
    # Conversion probability increases with stronger channels
    logit = -2.0
    if "search" in exposed:
        logit += 1.5
    if "display" in exposed:
        logit += 0.8
    if "browse" in exposed:
        logit += 0.3
    if "email" in exposed:
        logit += 0.1
    prob = 1.0 / (1.0 + np.exp(-logit))
    converted = int(np.random.rand() < prob)
    synthetic_journeys.append((frozenset(exposed), converted))

# Train the value function
value_fn = train_value_function(synthetic_journeys)

# Verify the trained model produces sensible probabilities
p_none = 0.0  # by definition
p_search = value_fn(frozenset({"search"}))
p_display = value_fn(frozenset({"display"}))
p_both = value_fn(frozenset({"search", "display"}))

print(f"P(conv | {{}}) = {p_none:.3f}")
print(f"P(conv | {{search}}) = {p_search:.3f}")
print(f"P(conv | {{display}}) = {p_display:.3f}")
print(f"P(conv | {{search, display}}) = {p_both:.3f}")

assert p_search > p_display, "Search should have higher lift than display"
assert p_both > p_search, "Both channels should beat search alone"
assert p_both > p_display, "Both channels should beat display alone"
print("\n✓ Trained value function produces sensible monotonic probabilities")

# Now compute Shapley values using the trained value_fn
result_e2e = exact_shapley_attribution(
    channels=["search", "display", "browse"],
    value_fn=value_fn,
    conversion_revenue=100.0,
)

total_e2e = sum(result_e2e.values())
print(f"\nShapley attribution (revenue=$100):")
for ch, credit in sorted(result_e2e.items(), key=lambda x: -x[1]):
    print(f"  {ch:>8s}: ${credit:>6.2f}  ({credit/total_e2e*100:.1f}%)")
print(f"  {'Total':>8s}: ${total_e2e:>6.2f}")

assert abs(total_e2e - 100.0) < 0.01, "Credits should sum to revenue"
assert result_e2e["search"] > result_e2e["display"] > result_e2e["browse"], \
    "Credit order should be search > display > browse"
print("\n✓ End-to-end pipeline: train value_fn → Shapley attribution works correctly")

P(conv | {}) = 0.000
P(conv | {search}) = 0.390
P(conv | {display}) = 0.228
P(conv | {search, display}) = 0.583

✓ Trained value function produces sensible monotonic probabilities

Shapley attribution (revenue=$100):
    search: $ 55.14  (55.1%)
   display: $ 29.31  (29.3%)
    browse: $ 15.54  (15.5%)
     Total: $100.00

✓ End-to-end pipeline: train value_fn → Shapley attribution works correctly


In [10]:
import numpy as np
from math import factorial
from itertools import combinations
from typing import Callable

def exact_shapley_attribution(
    channels: list[str],
    value_fn: Callable[[frozenset[str]], float],
    conversion_revenue: float,
) -> dict[str, float]:
    """Compute exact Shapley values for channel-level attribution.

    Feasible when len(channels) <= 6 (at most 64 coalition evaluations).

    Args:
        channels: List of channel names present in the user journey
            (e.g., ["search", "display", "browse"]).
        value_fn: Function mapping a frozenset of channels to conversion
            probability P(conversion | exposed to these channels).
        conversion_revenue: Total revenue from the conversion event.

    Returns:
        Dictionary mapping each channel to its attributed revenue share.
    """
    n = len(channels)
    shapley_values: dict[str, float] = {}

    for i, channel in enumerate(channels):
        others = [c for c in channels if c != channel]
        phi = 0.0

        for size in range(0, n):
            for subset in combinations(others, size):
                coalition = frozenset(subset)
                coalition_with_i = coalition | {channel}
                marginal = value_fn(coalition_with_i) - value_fn(coalition)
                weight = (
                    factorial(size)
                    * factorial(n - size - 1)
                    / factorial(n)
                )
                phi += weight * marginal

        shapley_values[channel] = phi

    # Normalize to sum to conversion_revenue
    total_phi = sum(shapley_values.values())
    if total_phi > 0:
        shapley_values = {
            ch: (phi / total_phi) * conversion_revenue
            for ch, phi in shapley_values.items()
        }

    return shapley_values

print("exact_shapley_attribution loaded successfully.")

exact_shapley_attribution loaded successfully.


In [11]:
# --- Test 6a: 2-channel hand-computed Shapley values ---
# With search and display, and a simple additive value function:
#   v({}) = 0.0, v({search}) = 0.3, v({display}) = 0.1, v({search, display}) = 0.5
# 
# Shapley for search:
#   phi(search) = 0.5 * [v({search}) - v({})] + 0.5 * [v({search,display}) - v({display})]
#               = 0.5 * (0.3 - 0.0) + 0.5 * (0.5 - 0.1) = 0.15 + 0.20 = 0.35
#
# Shapley for display:
#   phi(display) = 0.5 * [v({display}) - v({})] + 0.5 * [v({search,display}) - v({search})]
#                = 0.5 * (0.1 - 0.0) + 0.5 * (0.5 - 0.3) = 0.05 + 0.10 = 0.15
#
# Total phi = 0.35 + 0.15 = 0.50
# After normalization to revenue=$100:
#   search = (0.35 / 0.50) * 100 = $70
#   display = (0.15 / 0.50) * 100 = $30

value_table = {
    frozenset(): 0.0,
    frozenset({"search"}): 0.3,
    frozenset({"display"}): 0.1,
    frozenset({"search", "display"}): 0.5,
}

result = exact_shapley_attribution(
    channels=["search", "display"],
    value_fn=lambda s: value_table[s],
    conversion_revenue=100.0,
)

print(f"Search:  ${result['search']:.2f}")
print(f"Display: ${result['display']:.2f}")
print(f"Total:   ${sum(result.values()):.2f}")

assert abs(result["search"] - 70.0) < 0.01, f"Expected $70, got ${result['search']:.2f}"
assert abs(result["display"] - 30.0) < 0.01, f"Expected $30, got ${result['display']:.2f}"
assert abs(sum(result.values()) - 100.0) < 0.01, "Credits should sum to revenue"
print("\n✓ 2-channel Shapley matches hand-computed values (search=$70, display=$30)")

Search:  $70.00
Display: $30.00
Total:   $100.00

✓ 2-channel Shapley matches hand-computed values (search=$70, display=$30)


In [12]:
# --- Test 6b: Efficiency axiom — credits sum to revenue ---
# 3-channel example with a more complex value function

def value_fn_3ch(coalition: frozenset[str]) -> float:
    """Mock conversion probability for 3-channel model."""
    table = {
        frozenset(): 0.0,
        frozenset({"search"}): 0.25,
        frozenset({"display"}): 0.10,
        frozenset({"browse"}): 0.05,
        frozenset({"search", "display"}): 0.40,
        frozenset({"search", "browse"}): 0.35,
        frozenset({"display", "browse"}): 0.15,
        frozenset({"search", "display", "browse"}): 0.55,
    }
    return table[coalition]

result_3ch = exact_shapley_attribution(
    channels=["search", "display", "browse"],
    value_fn=value_fn_3ch,
    conversion_revenue=250.0,
)

total = sum(result_3ch.values())
for ch, credit in sorted(result_3ch.items(), key=lambda x: -x[1]):
    print(f"  {ch:>8s}: ${credit:>7.2f}  ({credit/total*100:.1f}%)")
print(f"  {'Total':>8s}: ${total:>7.2f}")

assert abs(total - 250.0) < 0.01, f"Credits should sum to $250, got ${total:.2f}"
print("\n✓ Efficiency axiom holds: 3-channel Shapley values sum to conversion revenue ($250)")

    search: $ 143.94  (57.6%)
   display: $  64.39  (25.8%)
    browse: $  41.67  (16.7%)
     Total: $ 250.00

✓ Efficiency axiom holds: 3-channel Shapley values sum to conversion revenue ($250)


In [13]:
# --- Test 6c: Null player axiom — a channel with zero marginal contribution gets $0 ---
# "dummy" adds nothing to any coalition

def value_fn_null(coalition: frozenset[str]) -> float:
    """dummy adds 0 marginal to every coalition."""
    effective = coalition - {"dummy"}
    table = {
        frozenset(): 0.0,
        frozenset({"search"}): 0.4,
        frozenset({"display"}): 0.2,
        frozenset({"search", "display"}): 0.6,
    }
    return table.get(effective, 0.0)

result_null = exact_shapley_attribution(
    channels=["search", "display", "dummy"],
    value_fn=value_fn_null,
    conversion_revenue=100.0,
)

print(f"Search:  ${result_null['search']:.2f}")
print(f"Display: ${result_null['display']:.2f}")
print(f"Dummy:   ${result_null['dummy']:.2f}")

assert abs(result_null["dummy"]) < 0.01, f"Dummy should get $0, got ${result_null['dummy']:.2f}"
assert result_null["search"] > result_null["display"], "Search should get more than display"
print("\n✓ Null player axiom holds: dummy channel gets $0 credit")

Search:  $66.67
Display: $33.33
Dummy:   $0.00

✓ Null player axiom holds: dummy channel gets $0 credit


In [14]:
# --- Test 6d: Symmetry axiom — identical channels get equal credit ---
# search_a and search_b have identical marginal contributions in all coalitions

def value_fn_sym(coalition: frozenset[str]) -> float:
    """search_a and search_b are interchangeable."""
    has_a = "search_a" in coalition
    has_b = "search_b" in coalition
    has_display = "display" in coalition
    # Treat search_a and search_b identically: each adds 0.2, no overlap
    base = 0.0
    if has_a or has_b:
        base = 0.2
    if has_a and has_b:
        base = 0.2  # no additional lift from second search channel
    if has_display:
        base += 0.15
    return base

result_sym = exact_shapley_attribution(
    channels=["search_a", "search_b", "display"],
    value_fn=value_fn_sym,
    conversion_revenue=100.0,
)

print(f"Search A: ${result_sym['search_a']:.2f}")
print(f"Search B: ${result_sym['search_b']:.2f}")
print(f"Display:  ${result_sym['display']:.2f}")

assert abs(result_sym["search_a"] - result_sym["search_b"]) < 0.01, \
    f"Symmetric channels should get equal credit: A=${result_sym['search_a']:.2f}, B=${result_sym['search_b']:.2f}"
print("\n✓ Symmetry axiom holds: search_a and search_b receive equal credit")

Search A: $28.57
Search B: $28.57
Display:  $42.86

✓ Symmetry axiom holds: search_a and search_b receive equal credit


## Summary (Sections 1–6)

All tests validate the core behaviors of Code Listings 12.1, 12.2, and 12.3:

| Test | Behavior Verified |
|------|-------------------|
| User Journey Replay | Full event sequence from Section 1.2 correctly attributes to most recent click |
| VIEW_LOOKBACK eviction | Unclicked impressions evicted after 1 day, kept within window |
| CLICK_LOOKBACK eviction | Clicked touchpoints evicted after 14 days |
| View-Through (in window) | Unclicked impression within 1 day qualifies |
| View-Through (out of window) | Unclicked impression beyond 1 day does not qualify |
| Click > Impression priority | Click beats a more recent view-only impression |
| Shapley hand-computed | 2-channel values match manual calculation |
| Shapley symmetry | Identical channels receive equal credit |
| End-to-end pipeline | Train value_fn from data → Shapley attribution produces correct ordering |
| Shapley efficiency | Credits sum exactly to conversion revenue |
| Shapley null player | Channel with zero marginal contribution gets $0 |

Sections 7–9 continue with PyFlink streaming join, IPS estimator, and pipeline monitoring tests.

## 7. PyFlink Streaming Attribution Join (Section 2.2)

The chapter's Section 2.2 describes a Flink-based streaming join for
near-real-time attribution.  The code below is a **production-shaped PyFlink
implementation** using `KeyedCoProcessFunction` and Flink `ValueState` for
per-user event buffers.  Because running a full Flink cluster in a notebook is
impractical, we test the `AttributionJoinFunction` logic directly (it is a plain
Python class whose state interface we mock with dictionaries).

In [1]:
# --------------------------------------------------------------------------
# PyFlink Streaming Attribution Join
#
# Production Flink jobs use KeyedCoProcessFunction to join two keyed streams
# (ad-events and conversions) sharing the same key (user_id).  Per-user state
# is stored in Flink's managed ValueState (backed by RocksDB) so it survives
# checkpoints and restarts.
#
# Below we define the real PyFlink types and the core join function.  For
# testability, the class also exposes a `_process_locally()` helper that runs
# the same logic against plain-dict state, so we can validate without a
# running Flink cluster.
# --------------------------------------------------------------------------

from __future__ import annotations
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Iterator

# ── Domain types ──────────────────────────────────────────────────────────

@dataclass(frozen=True)
class AdEvent:
    """An impression or click event from the ad-serving pipeline."""
    event_id: str
    user_id: str
    event_type: str               # "impression" | "click"
    impression_id: str            # links click → impression
    campaign_id: str
    creative_id: str
    timestamp: datetime

@dataclass(frozen=True)
class ConversionEvent:
    """A purchase or other conversion event."""
    conversion_id: str
    user_id: str
    revenue: float
    timestamp: datetime
    source: str = "online"        # "online" | "offline"

@dataclass
class AttributionResult:
    conversion_id: str
    user_id: str
    revenue: float
    credited_impression_id: str
    credited_campaign_id: str
    touch_type: str               # "click" | "view"
    data_completeness: str        # "preliminary" | "reconciled"


# ── Core join logic (the function Flink would call) ──────────────────────

class AttributionJoinFunction:
    """Windowed join matching conversions to ad events.

    In production this would extend ``KeyedCoProcessFunction``.  Two managed
    ``ValueState[list[AdEvent]]`` objects hold per-user impression and click
    buffers.  We implement the logic here against plain dicts so the same
    code can be unit-tested *and* wrapped in a Flink harness.

    Flink specifics handled outside this class:
      - KeyBy(user_id) partitions both streams
      - Checkpointing + 2PC to Kafka for exactly-once
      - Watermark generation (5 min online, 72 h offline)
      - Late-event side-output to a reconciliation topic
    """

    CLICK_WINDOW = timedelta(days=14)
    VIEW_WINDOW  = timedelta(days=1)

    def __init__(self) -> None:
        # Per-user state (in Flink: ValueState backed by RocksDB)
        self._impressions: dict[str, list[AdEvent]] = {}   # user → [events]
        self._clicks: dict[str, list[AdEvent]] = {}

    # -- stream-1 handler (ad events) ------------------------------------

    def process_ad_event(self, event: AdEvent) -> None:
        """Buffer an ad event for future join with conversions."""
        if event.event_type == "impression":
            self._impressions.setdefault(event.user_id, []).append(event)
        elif event.event_type == "click":
            self._clicks.setdefault(event.user_id, []).append(event)

    # -- stream-2 handler (conversions) ----------------------------------

    def process_conversion(
        self, conv: ConversionEvent
    ) -> AttributionResult | None:
        """Join a conversion with buffered ad events in the window.

        Priority: most-recent click within CLICK_WINDOW, falling back
        to most-recent impression within VIEW_WINDOW.
        """
        # Clicks within window
        clicks = [
            e for e in self._clicks.get(conv.user_id, [])
            if timedelta(0) <= conv.timestamp - e.timestamp <= self.CLICK_WINDOW
        ]
        if clicks:
            best = max(clicks, key=lambda e: e.timestamp)
            return AttributionResult(
                conversion_id=conv.conversion_id,
                user_id=conv.user_id,
                revenue=conv.revenue,
                credited_impression_id=best.impression_id,
                credited_campaign_id=best.campaign_id,
                touch_type="click",
                data_completeness="preliminary",
            )

        # View-through fallback
        views = [
            e for e in self._impressions.get(conv.user_id, [])
            if timedelta(0) <= conv.timestamp - e.timestamp <= self.VIEW_WINDOW
        ]
        if views:
            best = max(views, key=lambda e: e.timestamp)
            return AttributionResult(
                conversion_id=conv.conversion_id,
                user_id=conv.user_id,
                revenue=conv.revenue,
                credited_impression_id=best.impression_id,
                credited_campaign_id=best.campaign_id,
                touch_type="view",
                data_completeness="preliminary",
            )

        return None

    # -- timer-driven state expiry (Flink onTimer callback) --------------

    def expire_state(self, now: datetime) -> int:
        """Remove events older than the maximum attribution window.

        In Flink this runs via a processing-time timer registered per key.
        Returns the number of expired events (for metrics).
        """
        click_cutoff = now - self.CLICK_WINDOW
        view_cutoff  = now - self.VIEW_WINDOW
        expired = 0

        for uid in list(self._clicks):
            before = len(self._clicks[uid])
            self._clicks[uid] = [
                e for e in self._clicks[uid] if e.timestamp > click_cutoff
            ]
            expired += before - len(self._clicks[uid])
            if not self._clicks[uid]:
                del self._clicks[uid]

        for uid in list(self._impressions):
            before = len(self._impressions[uid])
            self._impressions[uid] = [
                e for e in self._impressions[uid] if e.timestamp > view_cutoff
            ]
            expired += before - len(self._impressions[uid])
            if not self._impressions[uid]:
                del self._impressions[uid]

        return expired


# ── PyFlink pipeline skeleton (for reference; not executed here) ─────────

PYFLINK_PIPELINE = """
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.datastream.functions import KeyedCoProcessFunction
from pyflink.common.typeinfo import Types
from pyflink.datastream.state import ValueStateDescriptor

class FlinkAttributionJoin(KeyedCoProcessFunction):
    '''Flink wrapper around AttributionJoinFunction.'''

    def open(self, ctx):
        desc = ValueStateDescriptor('ad_events', Types.PICKLED_BYTE_ARRAY())
        self.state = ctx.get_runtime_context().get_state(desc)
        self.join = AttributionJoinFunction()

    def process_element1(self, event, ctx):
        # ad-event stream
        self.join.process_ad_event(event)
        # register timer for state expiry
        ctx.timer_service().register_processing_time_timer(
            ctx.timestamp() + 14 * 86400 * 1000)

    def process_element2(self, conversion, ctx):
        # conversion stream
        result = self.join.process_conversion(conversion)
        if result:
            yield result

    def on_timer(self, ts, ctx):
        self.join.expire_state(datetime.fromtimestamp(ts / 1000))


env = StreamExecutionEnvironment.get_execution_environment()
env.enable_checkpointing(60_000)          # 60-second checkpoints
# ... Kafka source/sink configuration, keyBy(user_id), connect(), etc.
"""

print("AttributionJoinFunction loaded successfully.")
print(f"PyFlink pipeline skeleton: {len(PYFLINK_PIPELINE)} chars (reference only, not executed)")


AttributionJoinFunction loaded successfully.
PyFlink pipeline skeleton: 1288 chars (reference only, not executed)


In [2]:
# --- Test 7a: Click attribution within window ---
join = AttributionJoinFunction()
t0 = datetime(2026, 3, 8, 12, 0, 0)

join.process_ad_event(AdEvent(
    event_id="ev1", user_id="u1", event_type="impression",
    impression_id="imp_1", campaign_id="camp_A",
    creative_id="cr_1", timestamp=t0 - timedelta(hours=2, minutes=1),
))
join.process_ad_event(AdEvent(
    event_id="ev2", user_id="u1", event_type="click",
    impression_id="imp_1", campaign_id="camp_A",
    creative_id="cr_1", timestamp=t0 - timedelta(hours=2),
))

result = join.process_conversion(ConversionEvent(
    conversion_id="conv_1", user_id="u1", revenue=50.0, timestamp=t0,
))

assert result is not None
assert result.touch_type == "click"
assert result.credited_impression_id == "imp_1"
assert result.credited_campaign_id == "camp_A"
assert result.revenue == 50.0
print(f"7a: Attributed conv_1 via {result.touch_type} on {result.credited_impression_id}")
print("✓ Click within CLICK_WINDOW correctly attributed")


7a: Attributed conv_1 via click on imp_1
✓ Click within CLICK_WINDOW correctly attributed


In [3]:
# --- Test 7b: View-through fallback (no click in window) ---
join2 = AttributionJoinFunction()
t0 = datetime(2026, 3, 8, 12, 0, 0)

# Only an impression, no click
join2.process_ad_event(AdEvent(
    event_id="ev10", user_id="u2", event_type="impression",
    impression_id="imp_10", campaign_id="camp_B",
    creative_id="cr_2", timestamp=t0 - timedelta(hours=20),
))

result_vt = join2.process_conversion(ConversionEvent(
    conversion_id="conv_vt", user_id="u2", revenue=30.0, timestamp=t0,
))

assert result_vt is not None
assert result_vt.touch_type == "view"
assert result_vt.credited_impression_id == "imp_10"
print(f"7b: View-through attribution on {result_vt.credited_impression_id}")
print("✓ Impression within VIEW_WINDOW (20h) triggers view-through fallback")


7b: View-through attribution on imp_10
✓ Impression within VIEW_WINDOW (20h) triggers view-through fallback


In [4]:
# --- Test 7c: View-through outside window → no attribution ---
join3 = AttributionJoinFunction()
t0 = datetime(2026, 3, 8, 12, 0, 0)

# Impression 25 hours ago — outside VIEW_WINDOW (1 day)
join3.process_ad_event(AdEvent(
    event_id="ev20", user_id="u3", event_type="impression",
    impression_id="imp_20", campaign_id="camp_C",
    creative_id="cr_3", timestamp=t0 - timedelta(hours=25),
))

result_no = join3.process_conversion(ConversionEvent(
    conversion_id="conv_no", user_id="u3", revenue=20.0, timestamp=t0,
))

assert result_no is None
print("7c: No attribution for impression outside VIEW_WINDOW (25h > 1 day)")
print("✓ Expired impressions are not attributed")


7c: No attribution for impression outside VIEW_WINDOW (25h > 1 day)
✓ Expired impressions are not attributed


In [5]:
# --- Test 7d: Click beats impression (priority) ---
join4 = AttributionJoinFunction()
t0 = datetime(2026, 3, 8, 12, 0, 0)

# An impression 30 seconds ago (no click) — very recent
join4.process_ad_event(AdEvent(
    event_id="ev30", user_id="u4", event_type="impression",
    impression_id="imp_30", campaign_id="camp_D",
    creative_id="cr_4", timestamp=t0 - timedelta(seconds=30),
))

# A click 2 hours ago — older but should win
join4.process_ad_event(AdEvent(
    event_id="ev31", user_id="u4", event_type="click",
    impression_id="imp_31", campaign_id="camp_E",
    creative_id="cr_5", timestamp=t0 - timedelta(hours=2),
))

result_pri = join4.process_conversion(ConversionEvent(
    conversion_id="conv_pri", user_id="u4", revenue=80.0, timestamp=t0,
))

assert result_pri is not None
assert result_pri.touch_type == "click"
assert result_pri.credited_impression_id == "imp_31"
print(f"7d: Attributed to {result_pri.touch_type} on {result_pri.credited_impression_id} (camp_E)")
print("✓ Click (2h ago) beats impression (30s ago) — click priority enforced")


7d: Attributed to click on imp_31 (camp_E)
✓ Click (2h ago) beats impression (30s ago) — click priority enforced


In [6]:
# --- Test 7e: State expiry removes old events ---
join5 = AttributionJoinFunction()
t0 = datetime(2026, 1, 1, 12, 0, 0)

# Add impression + click at t0
join5.process_ad_event(AdEvent(
    event_id="ev40", user_id="u5", event_type="impression",
    impression_id="imp_40", campaign_id="camp_F",
    creative_id="cr_6", timestamp=t0,
))
join5.process_ad_event(AdEvent(
    event_id="ev41", user_id="u5", event_type="click",
    impression_id="imp_40", campaign_id="camp_F",
    creative_id="cr_6", timestamp=t0 + timedelta(seconds=10),
))

assert len(join5._impressions.get("u5", [])) == 1
assert len(join5._clicks.get("u5", [])) == 1

# Expire after 15 days — both should be removed
expired = join5.expire_state(t0 + timedelta(days=15))

assert expired == 2, f"Expected 2 expired events, got {expired}"
assert "u5" not in join5._impressions
assert "u5" not in join5._clicks
print(f"7e: Expired {expired} events after 15 days")
print("✓ expire_state() removes clicks past CLICK_WINDOW and impressions past VIEW_WINDOW")


7e: Expired 2 events after 15 days
✓ expire_state() removes clicks past CLICK_WINDOW and impressions past VIEW_WINDOW


## 8. Code Listing 12.4 — Inverse Propensity Scoring (IPS) Estimator (Section 3.3)

Tests for `ips_estimate` from Code Listing 12.4. We verify:
- **8a.** Identical policies → estimate equals the mean observed reward
- **8b.** Clipping caps extreme importance weights
- **8c.** Target policy that favors high-reward actions → higher estimate
- **8d.** All-zero rewards → zero estimate
- **8e.** Standard error decreases with more samples

In [1]:
# --- Load Code Listing 12.4: IPS Estimator ---
import numpy as np
from typing import NamedTuple

class LoggedEvent(NamedTuple):
    context: np.ndarray      # feature vector (user, query, page)
    action: int              # 0-based index into the action space
    reward: float            # observed outcome (revenue)
    logging_prob: float      # P(action | context) under logging policy

def ips_estimate(
    events: list[LoggedEvent],
    target_policy_probs: np.ndarray,  # shape: (n_events, n_actions)
    clip_ratio: float = 100.0
) -> tuple[float, float]:
    """Compute clipped IPS estimate of target policy value.

    Args:
        events: Historical logged events with logging probabilities.
        target_policy_probs: P(action | context) under the target policy
            for each event, shape (n_events, n_actions).
        clip_ratio: Maximum importance weight to reduce variance.

    Returns:
        Tuple of (estimated_value, standard_error).
    """
    n = len(events)
    weights = np.zeros(n)

    for i, event in enumerate(events):
        target_prob = target_policy_probs[i, event.action]
        ratio = target_prob / max(event.logging_prob, 1e-8)
        weights[i] = min(ratio, clip_ratio) * event.reward

    estimate = np.mean(weights)
    se = np.std(weights, ddof=1) / np.sqrt(n)
    return estimate, se

print("✓ Code Listing 12.4 loaded")

✓ Code Listing 12.4 loaded


In [2]:
# --- Test 8a: Identical policies → estimate = mean reward ---
# When target == logging, every importance weight is 1.0,
# so the IPS estimate should equal the simple mean of rewards.
np.random.seed(42)
n_events, n_actions = 100, 5

events_8a = []
target_probs_8a = np.zeros((n_events, n_actions))
for i in range(n_events):
    # Random softmax distribution (same for both policies)
    logits = np.random.randn(n_actions)
    probs = np.exp(logits) / np.exp(logits).sum()
    action = np.random.choice(n_actions, p=probs)
    reward = np.random.uniform(0, 10)
    events_8a.append(LoggedEvent(
        context=np.random.randn(3),
        action=action,
        reward=reward,
        logging_prob=probs[action],
    ))
    target_probs_8a[i] = probs  # identical to logging

est, se = ips_estimate(events_8a, target_probs_8a)
mean_reward = np.mean([e.reward for e in events_8a])

assert abs(est - mean_reward) < 1e-10, f"Expected {mean_reward}, got {est}"
print(f"8a: IPS estimate = {est:.4f}, mean reward = {mean_reward:.4f}")
print("✓ Identical policies produce estimate equal to mean reward")

8a: IPS estimate = 4.6951, mean reward = 4.6951
✓ Identical policies produce estimate equal to mean reward


In [3]:
# --- Test 8b: Clipping caps extreme importance weights ---
# One event has logging_prob ≈ 0 → huge raw ratio.
# With clip_ratio=10, the weight should be capped.
events_8b = [
    LoggedEvent(context=np.array([1.0]), action=0, reward=5.0, logging_prob=0.001),
    LoggedEvent(context=np.array([1.0]), action=1, reward=3.0, logging_prob=0.5),
]
target_probs_8b = np.array([
    [0.8, 0.2],  # event 0: target strongly favors action 0
    [0.3, 0.7],  # event 1: target favors action 1
])

# Without clipping (high limit)
est_unclipped, _ = ips_estimate(events_8b, target_probs_8b, clip_ratio=1e6)
# With clipping at 10
est_clipped, _ = ips_estimate(events_8b, target_probs_8b, clip_ratio=10.0)

# Raw ratio for event 0: 0.8/0.001 = 800 → clipped to 10
# Clipped weight[0] = 10.0 * 5.0 = 50.0
# Raw ratio for event 1: 0.7/0.5 = 1.4 → not clipped
# Weight[1] = 1.4 * 3.0 = 4.2
# Clipped estimate = (50.0 + 4.2) / 2 = 27.1
expected_clipped = (10.0 * 5.0 + 1.4 * 3.0) / 2

assert abs(est_clipped - expected_clipped) < 1e-10, f"Expected {expected_clipped}, got {est_clipped}"
assert est_clipped < est_unclipped, "Clipping should reduce the estimate"
print(f"8b: Unclipped = {est_unclipped:.1f}, Clipped (ratio≤10) = {est_clipped:.1f}")
print("✓ Clipping caps extreme importance weights")

8b: Unclipped = 2002.1, Clipped (ratio≤10) = 27.1
✓ Clipping caps extreme importance weights


In [4]:
# --- Test 8c: Target that favors high-reward actions → higher estimate ---
# Logging policy is uniform; target policy concentrates on the best action.
np.random.seed(99)
n = 200
n_act = 4
rewards_by_action = [1.0, 2.0, 5.0, 0.5]  # action 2 is best

events_8c = []
target_probs_8c = np.zeros((n, n_act))
for i in range(n):
    action = np.random.randint(n_act)  # uniform logging
    events_8c.append(LoggedEvent(
        context=np.random.randn(3),
        action=action,
        reward=rewards_by_action[action],
        logging_prob=1.0 / n_act,  # uniform
    ))
    # Target policy: softmax with strong preference for action 2
    target_logits = np.array([0.0, 0.0, 5.0, 0.0])
    target_probs_8c[i] = np.exp(target_logits) / np.exp(target_logits).sum()

est_target, _ = ips_estimate(events_8c, target_probs_8c)
est_uniform, _ = ips_estimate(
    events_8c,
    np.full((n, n_act), 1.0 / n_act),  # uniform target = logging
)

assert est_target > est_uniform, (
    f"Target favoring best action should have higher value: {est_target:.2f} vs {est_uniform:.2f}"
)
print(f"8c: Uniform policy value = {est_uniform:.2f}, Smart target value = {est_target:.2f}")
print("✓ Target policy favoring high-reward actions produces higher estimate")

8c: Uniform policy value = 2.11, Smart target value = 4.63
✓ Target policy favoring high-reward actions produces higher estimate


In [5]:
# --- Test 8d: All-zero rewards → zero estimate ---
events_8d = [
    LoggedEvent(context=np.array([1.0]), action=0, reward=0.0, logging_prob=0.5),
    LoggedEvent(context=np.array([2.0]), action=1, reward=0.0, logging_prob=0.3),
    LoggedEvent(context=np.array([3.0]), action=0, reward=0.0, logging_prob=0.6),
]
target_probs_8d = np.array([
    [0.7, 0.3],
    [0.2, 0.8],
    [0.9, 0.1],
])

est_zero, se_zero = ips_estimate(events_8d, target_probs_8d)
assert est_zero == 0.0, f"Expected 0.0, got {est_zero}"
assert se_zero == 0.0, f"Expected SE 0.0, got {se_zero}"
print(f"8d: Estimate = {est_zero}, SE = {se_zero}")
print("✓ All-zero rewards produce zero estimate and zero SE")

8d: Estimate = 0.0, SE = 0.0
✓ All-zero rewards produce zero estimate and zero SE


In [6]:
# --- Test 8e: SE decreases with more samples ---
# Law of large numbers: SE ~ 1/sqrt(n), so doubling n should shrink SE.
np.random.seed(7)
n_actions = 3

def make_events(n):
    evts = []
    tgt = np.zeros((n, n_actions))
    for i in range(n):
        probs_log = np.array([0.5, 0.3, 0.2])
        probs_tgt = np.array([0.3, 0.5, 0.2])
        action = np.random.choice(n_actions, p=probs_log)
        evts.append(LoggedEvent(
            context=np.random.randn(2),
            action=action,
            reward=np.random.exponential(2.0),
            logging_prob=probs_log[action],
        ))
        tgt[i] = probs_tgt
    return evts, tgt

evts_small, tgt_small = make_events(100)
evts_large, tgt_large = make_events(10_000)

_, se_small = ips_estimate(evts_small, tgt_small)
_, se_large = ips_estimate(evts_large, tgt_large)

assert se_large < se_small, f"SE should decrease with n: {se_large:.4f} >= {se_small:.4f}"
print(f"8e: SE(n=100) = {se_small:.4f}, SE(n=10000) = {se_large:.4f}")
print(f"    Ratio = {se_small/se_large:.1f}x (theoretical ≈ 10x)")
print("✓ Standard error decreases with more samples")

8e: SE(n=100) = 0.1946, SE(n=10000) = 0.0234
    Ratio = 8.3x (theoretical ≈ 10x)
✓ Standard error decreases with more samples


## 9. Code Listing 12.5 — Pipeline Health Monitoring (Section 6.1)

Tests for `MeasurementPipelineMonitor` from Code Listing 12.5. We verify:
- **9a.** Volume anomaly detection fires when throughput drops >3σ below baseline
- **9b.** No false alarm when volume is within normal range
- **9c.** Join rate alert fires when rate drops below threshold
- **9d.** Latency P99 alert fires when SLA is breached
- **9e.** No alerts when all metrics are healthy

In [7]:
# --- Load Code Listing 12.5: Pipeline Health Monitor ---
import numpy as np
from dataclasses import dataclass
from datetime import datetime, timedelta

@dataclass
class PipelineMetrics:
    timestamp: datetime
    impressions_per_min: float
    clicks_per_min: float
    conversions_per_min: float
    join_rate: float                # fraction of conversions joined
    join_latency_p50_ms: float
    join_latency_p99_ms: float
    dedup_rate: float               # fraction of events deduplicated

class MeasurementPipelineMonitor:
    """Monitors attribution pipeline health and raises alerts.

    Compares real-time metrics against trailing 7-day baselines
    using z-score anomaly detection with configurable thresholds.
    """

    def __init__(
        self,
        volume_z_threshold: float = 3.0,
        join_rate_min: float = 0.50,
        latency_p99_max_ms: float = 60_000.0,   # 60s max join latency
        credit_sum_tolerance: float = 1e-4,
    ):
        self.volume_z_threshold = volume_z_threshold
        self.join_rate_min = join_rate_min
        self.latency_p99_max_ms = latency_p99_max_ms
        self.credit_sum_tolerance = credit_sum_tolerance
        # Trailing 7-day hourly baselines: hour -> list of observations
        self._baselines: dict[int, list[float]] = {h: [] for h in range(24)}

    def check_volume_anomaly(
        self, current: PipelineMetrics
    ) -> list[str]:
        """Detect sudden drops in event throughput using z-scores."""
        alerts: list[str] = []
        hour = current.timestamp.hour
        baseline = self._baselines.get(hour, [])

        if len(baseline) < 7:
            return alerts   # insufficient history

        mean = np.mean(baseline)
        std = np.std(baseline)
        if std == 0:
            return alerts

        z_score = (current.impressions_per_min - mean) / std
        if z_score < -self.volume_z_threshold:
            alerts.append(
                f"CRITICAL: Impression volume {current.impressions_per_min:.0f}/min "
                f"is {abs(z_score):.1f} std devs below baseline "
                f"({mean:.0f}/min). Possible ad server logging failure."
            )
        return alerts

    def check_join_health(
        self, current: PipelineMetrics
    ) -> list[str]:
        """Verify join rate and latency are within bounds."""
        alerts: list[str] = []

        if current.join_rate < self.join_rate_min:
            alerts.append(
                f"WARNING: Join rate {current.join_rate:.1%} below "
                f"threshold {self.join_rate_min:.1%}. "
                f"Check user ID resolution pipeline."
            )

        if current.join_latency_p99_ms > self.latency_p99_max_ms:
            alerts.append(
                f"WARNING: Join latency P99 {current.join_latency_p99_ms:.0f}ms "
                f"exceeds {self.latency_p99_max_ms:.0f}ms SLA. "
                f"Check Flink backpressure and checkpoint duration."
            )
        return alerts

print("✓ Code Listing 12.5 loaded")

✓ Code Listing 12.5 loaded


In [8]:
# --- Test 9a: Volume anomaly fires on >3σ drop ---
monitor = MeasurementPipelineMonitor(volume_z_threshold=3.0)

# Seed 7 days of baseline at hour=10: ~1000 imp/min with small variance
np.random.seed(42)
monitor._baselines[10] = [1000 + np.random.normal(0, 20) for _ in range(7)]

# Normal metrics template
def make_metrics(imp_per_min=1000.0, join_rate=0.70,
                 latency_p99=5000.0, hour=10):
    return PipelineMetrics(
        timestamp=datetime(2026, 3, 8, hour, 0, 0),
        impressions_per_min=imp_per_min,
        clicks_per_min=imp_per_min * 0.02,
        conversions_per_min=imp_per_min * 0.001,
        join_rate=join_rate,
        join_latency_p50_ms=latency_p99 * 0.3,
        join_latency_p99_ms=latency_p99,
        dedup_rate=0.02,
    )

# Sudden 60% drop → should fire
crashed = make_metrics(imp_per_min=400.0)
alerts_9a = monitor.check_volume_anomaly(crashed)
assert len(alerts_9a) == 1, f"Expected 1 alert, got {len(alerts_9a)}"
assert "CRITICAL" in alerts_9a[0]
print(f"9a: {alerts_9a[0]}")
print("✓ Volume anomaly correctly detected on >3σ drop")

# --- Test 9b: No false alarm when within normal range ---
normal = make_metrics(imp_per_min=990.0)
alerts_9b = monitor.check_volume_anomaly(normal)
assert len(alerts_9b) == 0, f"Expected 0 alerts, got {len(alerts_9b)}: {alerts_9b}"
print(f"\n9b: No alerts for volume={normal.impressions_per_min}/min (within baseline)")
print("✓ No false alarm when volume is within normal range")

9a: CRITICAL: Impression volume 400/min is 41.9 std devs below baseline (1010/min). Possible ad server logging failure.
✓ Volume anomaly correctly detected on >3σ drop

9b: No alerts for volume=990.0/min (within baseline)
✓ No false alarm when volume is within normal range


In [9]:
# --- Test 9c: Join rate alert fires below threshold ---
low_join = make_metrics(join_rate=0.35)
alerts_9c = monitor.check_join_health(low_join)
assert any("Join rate" in a for a in alerts_9c), f"Expected join rate alert, got {alerts_9c}"
print(f"9c: {alerts_9c[0]}")
print("✓ Join rate alert fires when rate (35%) drops below 50% threshold")

# --- Test 9d: Latency P99 alert fires when SLA breached ---
slow = make_metrics(latency_p99=90_000.0)
alerts_9d = monitor.check_join_health(slow)
assert any("latency" in a.lower() for a in alerts_9d), f"Expected latency alert, got {alerts_9d}"
print(f"\n9d: {alerts_9d[0]}")
print("✓ Latency P99 alert fires when 90s exceeds 60s SLA")

# --- Test 9e: No alerts when all metrics are healthy ---
healthy = make_metrics(imp_per_min=1010.0, join_rate=0.72, latency_p99=5000.0)
vol_alerts = monitor.check_volume_anomaly(healthy)
join_alerts = monitor.check_join_health(healthy)
all_alerts = vol_alerts + join_alerts
assert len(all_alerts) == 0, f"Expected 0 alerts for healthy metrics, got {all_alerts}"
print(f"\n9e: All checks passed with 0 alerts for healthy pipeline")
print("✓ No alerts when all metrics are within healthy bounds")

9c: WARNING: Join rate 35.0% below threshold 50.0%. Check user ID resolution pipeline.
✓ Join rate alert fires when rate (35%) drops below 50% threshold

9d: WARNING: Join latency P99 90000ms exceeds 60000ms SLA. Check Flink backpressure and checkpoint duration.
✓ Latency P99 alert fires when 90s exceeds 60s SLA

9e: All checks passed with 0 alerts for healthy pipeline
✓ No alerts when all metrics are within healthy bounds
